<a href="https://colab.research.google.com/github/FarabiOnAMission/Machine-Learning-Stuffs/blob/main/Singular_Value_Decomposition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [ ]:
def power_iteration(M,num_iters=100):
  #M is our transformation matrix, in SVD the thing is like A^T A = U Sigma V, so M is A^T A. so its a square matrix
  n = M.shape[1] # we find the number of cols of M
  v = np.random.randn(n) # v is a random vector with the shape of M's cols
  v = v/ np.linalg.norm(v) # we normalize it to 1

  #the process we take a random vector v at first, do power iteration re-initialize it every time with M@v then multiply by M again, then we will eventually get a vector that points to the largest direction of M , the largest direction M points

  for _ in range(num_iters):
    Mv = M @ v
    v = Mv / np.linalg.norm(Mv)

  #now M is just a connection of the largest pull in a direction
  #WE know find the eigenvalue of it
  eigenvalue = v @ M @ v
  # Mv = (lembda)v => v^T M v = v^T (lembda) v => v^T M v = lembda (v^T . v) => (v^T M v) = lembda //Because v is normalized, dotting with itself returns 1
  return eigenvalue,v


def svd_from_scratch(A, k=None):
  m, n = A.shape
  if k is None:
    k = min(m,n) #if the user didnt specify how many features he wants, then i would take k features the minimum number of m,n
  sigmas = []
  us = []
  vs = []

  A_residual = A.copy().astype(float)

  for _ in range(k):
    AtA = A_residual.T @ A_residual
    eigenvalue,v = power_iteration(AtA, num_iters = 200)

    if eigenvalue < 1e-10:
      break

    sigma = np.sqrt(eigenvalue)
    #A @ v = sigma . u =====> u = A @ v / sigma
    u = A_residual @ v / sigma

    sigmas.append(sigma)
    us.append(u)
    vs.append(v)

    A_residual = A_residual - sigma * np.outer(u, v) #Subtract the k-th feature we were working on

  U = np.column_stack(us) if us else np.empty((m, 0))
  S = np.array(sigmas)
  V = np.column_stack(vs) if vs else np.empty((n, 0))

  return U,S,V

In [ ]:
def svd_via_eig(A):
  #Compute the square matrix A^T A
  AtA = A.T @ A
  #Compute the eigenvalues and the eigenvector of AtA
  eigenvalues, V = np.linalg.eigh(AtA)

  #Sort them in descending order
  sorted_idx = np.argsort(eigenvalues)[::-1]
  eigenvalues = eigenvalues[sorted_idx]
  V = V[:, sorted_idx]
